# Summary statistics of dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
from pathlib import Path

In [ ]:
# Import processed data from csv
# Define the path to the processed data based on the current user
current_user = os.getlogin()
PROC_DIR = Path(rf"/home/{current_user}/local-share/processed")

# Read the processed data
data = pd.read_csv(PROC_DIR / "cleaned_original.csv") 

# For now, we use cleaned_original.csv as processed data. When it becomes final, we can change this to the final processed data file.

# Total number of students and drop-outs

In [ ]:
# Count total number of students and drop-outs
total_students = data['SINH_ID'].nunique()
total_dropouts = data[data['sdo5_degree_drop_out'] == 1]['SINH_ID'].nunique()
drop_out_rate = total_dropouts / total_students * 100

# Print the results
print(f"Total number of students: {total_students}")
print(f"Total number of drop-outs: {total_dropouts}")
print(f"Drop-out rate: {drop_out_rate:.2f}%")

## Number of students and drop-outs per year

In [ ]:
# Visualise number of continuing freshman and the number of drop-outs of 'sdo5_degree_drop_out' per sdo5_degree_COLLEGEJAAR
sns.countplot(x='sdo5_degree_COLLEGEJAAR', hue='sdo5_degree_drop_out', data=data)
plt.title('Number of Continuing Freshman and Drop-outs per College Year')
plt.xlabel('College Year')
plt.ylabel('Count')
plt.legend(title='', labels=['Continued studying', 'Dropped Out'])
plt.show()

# Counts and percentages (drop-out rate) per year
for year in sorted(data['sdo5_degree_COLLEGEJAAR'].unique()):
    year_data = data[data['sdo5_degree_COLLEGEJAAR'] == year]
    total_students = len(year_data)
    dropouts = year_data['sdo5_degree_drop_out'].sum()
    dropout_rate = (dropouts / total_students) * 100
    print(f"Year: {year}, Total Students: {total_students}, Drop-outs: {dropouts}, Dropout Rate: {dropout_rate:.2f}%")

## Number of drop-outs during the college year

In [ ]:
# Visualise drop-out numbers throughout the college year
data['dropout_date'] = pd.to_datetime(data['dropout_date'])
data['dropout_month'] = data['dropout_date'].dt.month
dropout_counts = data[data['sdo5_degree_drop_out'] == 1].groupby('dropout_month').size()

# Add column with percentage of drop-outs per month
dropout_counts = dropout_counts.reset_index(name='dropout_count')
dropout_counts['dropout_percentage'] = (dropout_counts['dropout_count'] / dropout_counts['dropout_count'].sum()) * 100

# Add column with college year month numbers
dropout_counts['college_year_month'] = dropout_counts['dropout_month'] - 8
dropout_counts.loc[dropout_counts['college_year_month'] <= 0, 'college_year_month'] += 12

# Add month names for better readability
month_names = {1: 'September', 2: 'October', 3: 'November', 4: 'December', 5: 'January', 6: 'February',
               7: 'March', 8: 'April', 9: 'May', 10: 'June', 11: 'July', 12: 'August'}
dropout_counts['month_name'] = dropout_counts['dropout_month'].map(month_names)

print(dropout_counts)

# Plot drop-out counts per month
sns.barplot(x='college_year_month', y='dropout_count', data=dropout_counts)
plt.title('Number of Drop-outs per Month')
plt.xlabel('Collegeyear month (numbered from September)')
plt.ylabel('Number of Drop-outs')
plt.show()

## Types of drop-out categories
Please note that drop-out categories are not mutually exclusive: e.g. a student who drops out without a propedeuse can also switch to another degree within the HU. 

In [ ]:
# Count students in each drop-out category:
# - temporary drop-out (sdo5_degree_drop_out_temporary)
# - switching program within HU (sdo5_degree_drop_out_to_other_degree_in_HU)
# - drop-out with propedeuse (sdo5_degree_drop_out_with_propedeuse)
# - drop-out without propedeuse (sdo5_degree_drop_out_without_propedeuse)
dropout_categories = [
    'sdo5_degree_drop_out_temporary',
    'sdo5_degree_drop_out_to_other_degree_in_HU',
    'sdo5_degree_drop_out_with_propedeuse',
    'sdo5_degree_drop_out_without_propedeuse'
]

dropout_categories_labels = [
    'Temporary drop-out',
    'Switching within HU',
    'Drop-out with propedeuse',
    'Drop-out without propedeuse'
]

# Visualize drop-out category counts in a table
dropout_counts = {category: data[category].sum() for category in dropout_categories}
dropout_df = pd.DataFrame(list(dropout_counts.items()), columns=['Drop-out category', 'Number of students'])
dropout_df['Drop-out category'] = dropout_categories_labels

# Add percentage column that computes the percentage of students in each drop-out category to the total number of drop-outs
dropout_df['Percentage'] = (dropout_df['Number of students'] / total_dropouts) * 100
print(dropout_df)

### Results
The above results show that students that drop-out with a propedeuse, or drop out temporarily (i.e. discontinue their studies for less than 1 year) are quite rare (<0.1% and < 2% of drop-outs respectively). This means that almost all students that drop-out do so without propedeuse (99%), and most students also do not switch to a different degree within the HU (79%). 

## Drop-out per degree

In [ ]:
# Look at the drop-out rate per degree (sdo5_degree_degree)
degree_dropout = data.groupby('sdo5_degree_degree')['sdo5_degree_drop_out'].agg(['sum', 'count'])
degree_dropout['dropout_rate'] = degree_dropout['sum'] / degree_dropout['count']
degree_dropout['dropout_rate_prc'] = degree_dropout['sum'] / degree_dropout['count'] * 100

# Visualize drop-out rate per degree in a vertical bar plot
degree_dropout = degree_dropout.sort_values(by='dropout_rate_prc', ascending=False) # Order data by drop-out rate
plt.figure(figsize=(12, 6))
sns.barplot(x=degree_dropout.index, y='dropout_rate_prc', data=degree_dropout.reset_index())
plt.title('Drop-out rate per HU degree')
plt.xlabel('Degree')
plt.ylabel('Drop-out (%)')
plt.xticks(rotation=90, ha='right')
plt.ylim(0, 100)
plt.axhline(y=drop_out_rate, color='r', linestyle='--', label='Average drop-out (%)')
plt.legend()    
plt.show()


## Statistics from V1

In [ ]:
data.columns

In [ ]:
print(data['Exit_Status'].value_counts())

In [ ]:
# Define a mapping for Exit_Status
exit_status_mapping = {
    'graduating': 0,
    'switching': 1,
    'dropping': 2
}

# Apply the mapping to replace categorical values with numbers
data['Exit_Status'] = data['Exit_Status'].map(exit_status_mapping)

# Show the first few rows to verify
print(data['Exit_Status'].value_counts())


In [ ]:
print(data['Exit_Status'].value_counts())

In [ ]:
# From the last result from the EDA.ipynb, we saw that Prior_Edu_Postcode, Prior_Edu_Country and Study_Prog_Exam_Completed have a corr >0.7, therefore we can drop them here.
# However, we are dropping them from the already one-hot-encoded data called feature_select

# Drop columns that start with 'Prior_Edu_Postcode' or 'Prior_Edu_Country'
cols_to_drop = [col for col in data.columns if col.startswith("Prior_Edu_Postcode") or col.startswith("Prior_Edu_Country")]

# Print columns to be removed
print("Columns to be removed:")
print(cols_to_drop)

# Drop the columns
data = data.drop(columns=cols_to_drop)

# Verify the remaining columns
print("Remaining columns:")
print(data.columns)



Print only the pairs of columns with correlation above 0.7

In [ ]:
# Convert Exit_Status to numeric values automatically (if not already done)
data['Exit_Status'] = data['Exit_Status'].astype('category').cat.codes  # This will assign 0,1,2 automatically

# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Find pairs with correlation above 0.8 (excluding self-correlation)
high_corr_pairs = []
threshold = 0.8

for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.append((col1, col2, corr_matrix.loc[col1, col2]))

# Remove duplicate pairs (since correlation is symmetric)
unique_high_corr_pairs = set(tuple(sorted(pair[:2])) + (pair[2],) for pair in high_corr_pairs)

# Print highly correlated column pairs
print("Highly Correlated Column Pairs (Above 80%):")
for col1, col2, corr in unique_high_corr_pairs:
    print(f"{col1} - {col2}: {corr:.2f}")


In [ ]:
del data['Enrollment_ID']

In [ ]:
data.to_csv(os.path.join("..", "data", "processed", "removed_high_corr.csv"), header=True)

In [ ]:
print(data['Exit_Status'].value_counts())

### Run a stats test to check for significance

In [ ]:
data.columns

In [ ]:
# Drop Enrollment_ID if present
if 'Enrollment_ID' in data.columns:
    data = data.drop(columns=['Enrollment_ID'])

# Ensure Exit_Status is numeric (already mapped previously)
y = data['Exit_Status']

# Use all other columns as predictors (X)
X = data.drop(columns=['Exit_Status'])

# Add constant term for intercept
X = sm.add_constant(X)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())

# Extract statistically significant features (p-value < 0.05)
significant_features = model.pvalues[model.pvalues < 0.05].index
print("\nSignificant Features (p < 0.05):")
print(significant_features)

# Extract non-significant features (p-value >= 0.05)
non_significant_features = model.pvalues[model.pvalues >= 0.05].index
print("\nNon-Significant Features (p >= 0.05):")
print(non_significant_features)


Highly statisfically significant features on Exiting include: 
Study_Prog_Exam_Completed
Study_Prog_Group_Size
Student_Enrollment_Gap
Study_Prog_Name( 19 features from it)
Prior_Edu_School_Location (22 features from it)

while the non significant features on Exiting include: 
Prior_Edu_Type
Study_Prog_Name
Prior_Edu_School_Location (more than 50 features from it)

Ambigious features: 
Study_Prog_Name
Prior_Edu_School_Location  

In [ ]:
# Set options to display all rows and columns
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.max_colwidth', None)    # Show full content of each column
pd.set_option('display.width', None)           # Adjust width for better readability
pd.set_option('display.expand_frame_repr', True)  # Expand the DataFrame across multiple lines if necessary
